In [40]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pc
import csv
import re
import sys
import numpy as np
import pyreadstat ### read sas7bdat into pandas df

In [44]:
# set paths 
fts_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.fts"
dat_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.dat"
out_path = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test_nopandas.parquet"
sas_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/2000/denominator/dnm2000.sas7bdat"
sas_out = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/sas.csv"

In [ ]:
# parse fts file to create dictionary of schema
def read_fts(fts_file):
    """
    Reads a .fts file and extracts column metadata into a structured dictionary.
    
    Args:
        fts_file (str): Path to the .fts file.
    
    Returns:
        dict: Parsed data with column headers as keys and row values as lists.
    """
    parsing = False  # Flag to start parsing after ----
    headers = [] 
    data_dict = {}
    header_lines = [] # fts headers
    column_widths = []  # width of fts columns
    
    with open(fts_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Find the header lines and the start of the table
    for i, line in enumerate(lines):
        line = line.rstrip()
        if '----' in line:
            parsing = True  # Start processing after this line
            if i > 2:
                header_lines = [l.rstrip() for l in lines[i-3:i]]  # headers are in the 3 lines before the '---'
            
            # Determine column widths of fts file by measuring dashes
            column_widths = [len(match.group()) for match in re.finditer(r'-+', line)]
            break
    
    # Extract column headers from three stacked lines
    start = 0
    for width in column_widths:
        column_header = ' '.join(line[start:start+width].strip() for line in header_lines).strip()
        headers.append(column_header)
        start += width + 1  # Move to next column start position
    
    # Initialize data dictionary with headers
    for header in headers:
        data_dict[header] = []
    
    # Process the data rows using fixed-width slicing based on column width
    for line in lines[i+1:]:  # Start from the first row after ----
        line = line.rstrip()
        if not line:
            continue
        
        # Stop parsing if the line contains "Note:" ### change this to make more robust 
        if "Note:" in line:
            break
        
        start = 0
        row_values = []
        for width in column_widths:
            row_values.append(line[start:start+width].strip())
            start += width + 1
        
        if len(row_values) == len(headers):
            for header, value in zip(headers, row_values):
                data_dict[header].append(value)
    
    return data_dict


In [ ]:
data_dict = read_fts(fts_path)
df = pd.DataFrame(data_dict)
df

First, attempt to simply parse dat file as csv using the schema from fts file

In [ ]:
def parse_dat_csv(dat_file, data_dict, output_csv, parquet=False, max_rows=100):
    """
    Parses a .dat file using the column widths and headers from data_dict and saves as CSV.
    Processes only the first `max_rows` rows.

    Args:
        dat_file (str): Path to the .dat file.
        data_dict (dict): Dictionary containing column metadata from read_fts.
        output_csv (str): Path to save the output CSV file.
        parquet (bool, optional): If True, also saves as Parquet. Default is False.
        max_rows (int, optional): Maximum number of rows to process. Default is 100.
    """
    keys = list(data_dict.keys())
    
    headers = data_dict[keys[2]]  # Extract headers from the 3rd key 
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    
    with open(dat_file, 'r', encoding='utf-8') as f, open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)  # Write headers
        
        for i, line in enumerate(f):
            if i >= max_rows:
                break  # Stop after processing max_rows
            
            line = line.rstrip()
            start = 0
            row_values = []
            
            for width in column_widths:
                row_values.append(line[start:start+width].strip())
                start += width
            
            writer.writerow(row_values)



In [ ]:
parse_dat_csv(dat_path,data_dict,out_path)
import pandas as pd 
pd.read_csv(out_path)


Parsing appears to be successful. Now will handle datatypes and converting to parquet.

In [ ]:
def change_dftypes(df, type_dict, verbose=False):
    """
    Converts column data types based on provided type mapping.
    
    Args:
        df (pd.DataFrame): Input DataFrame.
        type_dict (dict): Dictionary mapping column names to desired data types.
        verbose (bool): Whether to print type conversion messages.

    Returns:
        pd.DataFrame: DataFrame with updated types.
    """
    for col in df.columns:
        
        dtype = type_dict[col]

        if dtype == 'NUM':
            if verbose: print(f"{col}: CHAR --> NUM")
            df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')

        elif dtype == 'DATE':
            if verbose: print(f"{col}: CHAR --> DATE")
            df[col] = pd.to_datetime(df[col].str.strip(), errors='coerce')

        else:  # Default to string (CHAR)
            df[col] = df[col].str.strip().replace('', np.nan)
            if verbose: print(f"{col}: CHAR")

    return df


def parse_dat_parq_csv(dat_file, data_dict, output_csv, parquet=False, max_rows=20, verbose=False):
    """
    Parses a .dat file using the column widths and headers from data_dict, converts data types, and saves as CSV.
    
    Args:
        dat_file (str): Path to the .dat file.
        data_dict (dict): Dictionary containing column metadata from read_fts.
        output_csv (str): Path to save the output CSV file.
        parquet (bool): If True, also saves as Parquet. Default is False.
        max_rows (int): Maximum number of rows to process. Default is 100.
        verbose (bool): Whether to print debugging info.

    Returns:
        pd.DataFrame: Processed DataFrame.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    keys = list(data_dict.keys())

    headers = data_dict[keys[2]]  # Extract headers from the 3rd key
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    data_types = dict(zip(headers, data_dict[keys[3]]))  # Extract data types into a dictionary

    data = {header: [] for header in headers}  # Initialize storage

    # Read and parse .dat file
    with open(dat_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_rows:
                break  # Stop after max_rows
            
            line = line.rstrip()
            start = 0
            row_values = []
            
            for header, width in zip(headers, column_widths):
                row_values.append(line[start:start+width].strip())
                start += width

            for header, value in zip(headers, row_values):
                data[header].append(value)

    df = pd.DataFrame(data)

    # Apply type casting
    df = change_dftypes(df, data_types, verbose=verbose)

    df.to_csv(output_csv, index=False)
    
    if parquet:
        table = pa.Table.from_pandas(df)
        pq.write_table(table, output_csv.replace('.csv', '.parquet'))

    return df


In [ ]:
parse_dat_parq_csv(dat_path, data_dict, out_path, parquet=False, max_rows=1000, verbose=False)

In [ ]:
table = pq.read_table("/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test.parquet")
csv_table = pd.read_csv("/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test.csv")
csv_table

In [ ]:
print(table)

Seems like this methodology works, but it is reliant on pandas. Try to develop without dependence on pandas for more efficient processing 
### No pandas parsing

something weird going on with the dates...

In [ ]:
# reformat dates to work w numpy 
# can we read a fwf file efficiently without having to parse line by line? 
# retain csv files (need to update numpy function to do this)
# before 2011 we may not have resdac raw files? 
## parse .sas7bdat files for before 2011 
# look @ this: https://github.com/NSAPH-Data-Processing/legacy_mbsf_mortality_denom/blob/main/src/denom.py
# look @ 

# would be nice
## parsing by column 
## read file by chunk, in a byte sequence

### run on the 2015 data once complete

Using only pyarrow and numpy!!!

Parse dat file column wise instead of rowwise 

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pc
import numpy as np
from datetime import datetime

def change_dftypes_pyarrow(data, type_dict, verbose=False):
    """
    Converts column data types based on provided type mapping using PyArrow.
    """
    for col, values in data.items():
        dtype = type_dict.get(col, "CHAR")
        
        if dtype == 'NUM':
            if verbose: print(f"{col}: CHAR --> NUM")
            data[col] = pa.array([float(v) if v.replace('.', '', 1).isdigit() else None for v in values], type=pa.float64())
        
        elif dtype == 'DATE':
            if verbose: print(f"{col}: CHAR --> DATE")
            data[col] = pa.array([datetime.strptime(v, "%Y%m%d") if v else None for v in values], type=pa.date64())
        
        else:  # Default to string (CHAR)
            data[col] = pa.array([v.strip() if v.strip() else None for v in values], type=pa.string())
            if verbose: print(f"{col}: CHAR")
    
    return data

def parse_dat_parq_csv_pyarrow(dat_file, data_dict, output_csv, parquet=False, max_rows=20, verbose=False):
    """
    Parses a .dat file column-wise using the column widths and headers from data_dict, converts data types, and saves as CSV or Parquet.
    """
    keys = list(data_dict.keys())
    headers = data_dict[keys[2]]  # Extract headers from the 3rd key
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    data_types = dict(zip(headers, data_dict[keys[3]]))  # Extract data types into a dictionary
    
    data = {header: [] for header in headers}  # Initialize storage
    
    # Read and parse .dat file column by column
    with open(dat_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:max_rows]  # Read all lines up to max_rows
        
        for col_idx, (header, width) in enumerate(zip(headers, column_widths)):
            if verbose: print(f"Processing column: {header}")
            
            col_values = [line[sum(column_widths[:col_idx]):sum(column_widths[:col_idx]) + width].strip() for line in lines]
            data[header] = col_values
    
    # Apply type casting
    data = change_dftypes_pyarrow(data, data_types, verbose=verbose)
    
    # Convert to PyArrow table
    table = pa.table(data)
    
    # Write CSV
    pc.write_csv(table, output_csv)
    
    # Write Parquet if required
    if parquet:
        pq.write_table(table, output_csv.replace('.csv', '.parquet'))
    
    return table


In [ ]:
parse_dat_parq_csv_pyarrow(dat_path, data_dict, out_path, parquet=True, max_rows=20, verbose=False)

process sas7bdat file

In [45]:
import pyreadstat
import pyarrow as pa

def process_sas_columnwise(sas_file, output_csv, max_rows=20, parquet=True, verbose=False):
    """
    Process a SAS file column-wise and save it as CSV or Parquet without using pandas.
    """
    # Read the .sas7bdat file as dictionary instead of default pandas df
    data, meta = pyreadstat.read_sas7bdat(sas_file, row_offset=0, row_limit=max_rows, output_format='dict')  
    
    headers = meta.column_names  # Get column names
    column_types = meta.readstat_variable_types  # Get column types (for type casting)

    # Convert column-wise
    for col, dtype in column_types.items():
        values = data[col]  # Extract column values
        
        if col == 'DOB': 
            data[col] = pa.array(
                [datetime.strptime(str(int(v)), "%Y%m%d") if v else None for v in values], 
                type=pa.date64()
            )
            if verbose: print(f"{col}: Converted SAS numeric date to PyArrow date64")
            
        elif dtype == 'double':
            if verbose: print(f"{col}: double --> float")
            data[col] = pa.array(values, type=pa.float64())
        
        else:  # Default to string (CHAR)
            data[col] = pa.array(values, type=pa.string())
            if verbose: print(f"{col}: CHAR")
    
    # Convert to PyArrow table
    table = pa.table(data)
    
    # Write CSV
    pc.write_csv(table, output_csv)
    
    # Write Parquet if required
    if parquet:
        pq.write_table(table, output_csv.replace(".csv", ".parquet"))
    
    return table

# Example usage:
process_sas_columnwise(sas_path,sas_out, max_rows=20, parquet=True)


pyarrow.Table
STATE: string
ZIPCODE: double
DOB: date64[ms]
SEX: string
RACE: string
AGE: double
ORIG_ENT: string
CUR_ENT: string
ESRD_IND: string
MCSTATUS: string
PRTATERM: string
PRTBTERM: string
MC_ENT: string
HMOIND: string
HICOVG: double
SMICOVG: double
HMOCOVG: double
BUYCOVG: double
DODFLAG: string
BEF_DOD: double
ENROLYR: double
FIVE_PERCENT_FLAG: double
Intbid: string
----
STATE: [["30","15","22","30","30",...,"30","31","30","10","30"]]
ZIPCODE: [[38940613,461402438,16045163,34314513,33031127,...,38852453,78012525,30422425,347120221,33018642]]
DOB: [[1918-04-02,1908-12-09,1917-06-24,1917-05-04,1916-08-09,...,1915-09-28,1913-04-21,1932-12-05,1911-08-02,1916-10-09]]
SEX: [["2","2","1","2","2",...,"1","1","2","1","1"]]
RACE: [["1","1","1","1","1",...,"1","1","1","1","1"]]
AGE: [[81,91,82,82,83,...,84,86,67,88,83]]
ORIG_ENT: [["0","0","0","0","0",...,"0","0","1","0","0"]]
CUR_ENT: [["0","0","0","0","0",...,"0","0","1","0","0"]]
ESRD_IND: [["0","0","0","0","0",...,"0","0","0","0","

In [46]:
pd.read_csv("/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/sas.csv")

,STATE,ZIPCODE,DOB,SEX,RACE,AGE,ORIG_ENT,CUR_ENT,ESRD_IND,MCSTATUS,...,HMOIND,HICOVG,SMICOVG,HMOCOVG,BUYCOVG,DODFLAG,BEF_DOD,ENROLYR,FIVE_PERCENT_FLAG,Intbid
0,30,38940613,1918-04-02,2,1,81,0,0,0,10,...,000000000000,12,12,0,0,V,20010122,0,0,A00000001
1,15,461402438,1908-12-09,2,1,91,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,021814391
2,22,16045163,1917-06-24,1,1,82,0,0,0,10,...,CCCCCCCCCCCC,12,12,12,0,NaN,0,0,0,035833695
3,30,34314513,1917-05-04,2,1,82,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,020378499
4,30,33031127,1916-08-09,2,1,83,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,007308952
5,30,33031201,1913-02-22,2,1,86,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,041788293
6,10,341108614,1917-06-27,2,1,82,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,038994792
7,22,26630000,1906-11-07,1,1,93,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,009390097
8,30,33014445,1918-02-01,1,1,81,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,009540602
9,30,33010000,1911-12-12,1,1,88,0,0,0,10,...,000000000000,12,12,0,0,NaN,0,0,0,017557872


Editing function to run slurm job to chunk the data rowwise

In [29]:
# print(df.head())
#print(meta.column_names)
# print(meta.column_labels)
# print(meta.column_names_to_labels)
print(meta.number_rows)
print(meta.readstat_variable_types)
# print(meta.file_label)
# print(meta.file_encoding)

41587217
{'STATE': 'string', 'ZIPCODE': 'double', 'DOB': 'double', 'SEX': 'string', 'RACE': 'string', 'AGE': 'double', 'ORIG_ENT': 'string', 'CUR_ENT': 'string', 'ESRD_IND': 'string', 'MCSTATUS': 'string', 'PRTATERM': 'string', 'PRTBTERM': 'string', 'MC_ENT': 'string', 'HMOIND': 'string', 'HICOVG': 'double', 'SMICOVG': 'double', 'HMOCOVG': 'double', 'BUYCOVG': 'double', 'DODFLAG': 'string', 'BEF_DOD': 'double', 'ENROLYR': 'double', 'FIVE_PERCENT_FLAG': 'double', 'Intbid': 'string'}
